In [0]:
%sql
use catalog dkushari_uc;
create schema if not exists dkushari_uc.metric_view_demo;
use dkushari_uc.metric_view_demo;

In [0]:
%sql
create table if not exists customer deep clone samples.tpch.customer;
create table if not exists orders deep clone samples.tpch.orders;
create table if not exists lineitem deep clone samples.tpch.lineitem;
create table if not exists part deep clone samples.tpch.part;
create table if not exists supplier deep clone samples.tpch.supplier;
create table if not exists partsupp deep clone samples.tpch.partsupp;
create table if not exists nation deep clone samples.tpch.nation;
create table if not exists region deep clone samples.tpch.region;

source_table_size,source_num_of_files,num_of_synced_transactions,num_removed_files,num_copied_files,removed_files_size,copied_files_size
1632,1,null,0,1,0,1632


In [0]:
%sql
create schema if not exists metric_view_schema;

In [0]:
%sql
use dkushari_uc.metric_view_schema;
DROP VIEW IF EXISTS demo_orders_metric_view;
DROP VIEW IF EXISTS demo_lineitem_metric_view;

In [0]:
# version: 1.1

# source: dkushari_uc.metric_view_demo.orders

# filter: o_orderdate > '1990-01-01'

# dimensions:
#   - name: Order Year
#     expr: "DATE_TRUNC('YEAR', o_orderdate)"
#   - name: Order Month
#     expr: "date_format(o_orderdate, 'MMMM')"
#   - name: Order Month Date
#     expr: "DATE_TRUNC('MONTH', o_orderdate)"
#     format:
#       type: date
#       date_format: locale_short_month
#   - name: Order Status
#     expr: |-
#       case
#         when o_orderstatus = 'O' then 'Open'
#         when o_orderstatus = 'P' then 'Processing'
#         when o_orderstatus = 'F' then 'Fulfilled'
#       end
#   - name: Order Priority
#     expr: "split(o_orderpriority, '-')[1]"

# measures:
#   - name: Order Count
#     expr: count(1)
#   - name: Total Revenue
#     expr: SUM(o_totalprice)
#   - name: Total Revenue per Customer
#     expr: SUM(o_totalprice) / count(distinct o_custkey)
#   - name: Total Revenue for Open Orders
#     expr: SUM(o_totalprice) filter (where o_orderstatus='O')
#   - name: Total Revenue Prior Year
#     expr: SUM(o_totalprice)
#     window:
#       - order: Order Year
#         semiadditive: last
#         range: trailing 1 year
#   - name: Total Revenue Prior Month
#     expr: SUM(o_totalprice)
#     window:
#       - order: Order Month Date
#         semiadditive: last
#         range: trailing 1 month
#   - name: Revenue MoM Difference
#     expr: MEASURE(`Total Revenue`) - MEASURE(`Total Revenue Prior Month`)

In [0]:
%sql
use dkushari_uc.metric_view_schema;

In [0]:
%sql
describe table demo_orders_metric_view;

col_name,data_type,comment,metadata
Order Month,string,null,null
Order Month Date,timestamp,null,"{""format"":{""date"":{""date_format"":""LOCALE_SHORT_MONTH""}}}"
Order Status,string,null,null
Order Priority,string,null,null
Order Count,bigint measure,null,null
Total Revenue,"decimal(28,2) measure",null,null
Total Revenue per Customer,"decimal(38,12) measure",null,null
Total Revenue for Open Orders,"decimal(28,2) measure",null,null
Total Revenue Prior Month,"decimal(28,2) measure",null,null
Revenue MoM Difference,"decimal(29,2) measure",null,null


In [0]:
%sql
DESCRIBE TABLE EXTENDED demo_orders_metric_view;

col_name,data_type,comment,metadata
Order Month,string,null,null
Order Month Date,timestamp,null,"{""format"":{""date"":{""date_format"":""LOCALE_SHORT_MONTH""}}}"
Order Status,string,null,null
Order Priority,string,null,null
Order Count,bigint measure,null,null
Total Revenue,"decimal(28,2) measure",null,null
Total Revenue per Customer,"decimal(38,12) measure",null,null
Total Revenue for Open Orders,"decimal(28,2) measure",null,null
Total Revenue Prior Month,"decimal(28,2) measure",null,null
Revenue MoM Difference,"decimal(29,2) measure",null,null


###Evaluate 3 listed measures, using the metric view definition, and aggregate over Order Month and Order Status and sort by Order Month.

In [0]:
%sql
SELECT
    `Order Month Date` as current_month,
    `Order Status` as order_status,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Month`) as total_revenue_prior_month,
    MEASURE(`Revenue MoM Difference`) as revenue_mom_difference,
    MEASURE(`Total Revenue per Customer`) as total_revenue_per_customer,
    MEASURE(`Order Count`) as order_count
  FROM
    demo_orders_metric_view
  WHERE date_format(`Order Month Date`, 'MMMM') = :month
  GROUP BY ALL
  ORDER BY `Order Month Date` ASC

current_month,order_status,total_revenue,total_revenue_prior_month,revenue_mom_difference,total_revenue_per_customer,order_count
1992-04-01T00:00:00.000Z,Fulfilled,14154839483.68,14471982842.46,-317143358.78,167022.696508236183,93774
1993-04-01T00:00:00.000Z,Fulfilled,14125875685.61,14641206574.63,-515330889.02,167243.357276085387,93450
1994-04-01T00:00:00.000Z,Fulfilled,14191363315.44,14584304371.16,-392941055.72,167319.412792869270,93739
1995-04-01T00:00:00.000Z,Fulfilled,1157995419.89,4696391540.49,-3538396120.60,80405.181217192057,14647
1995-04-01T00:00:00.000Z,Processing,12022198561.63,9654834249.67,2367364311.96,196136.692415857737,65910
1995-04-01T00:00:00.000Z,Open,963981022.70,223277879.71,740703142.99,75630.081806056802,12927
1996-04-01T00:00:00.000Z,Open,14082461114.97,14564998783.77,-482537668.80,167075.516265304670,93197
1997-04-01T00:00:00.000Z,Open,14159948573.95,14597537077.94,-437588503.99,167482.182172426845,93608
1998-04-01T00:00:00.000Z,Open,14147094619.98,14618401602.72,-471306982.74,166830.913336006321,93779


In [0]:
%sql
SELECT
    `Order Status` as order_status,
    MEASURE(`Order Count`) as order_count,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Month`) as total_revenue_prior_month,
    MEASURE(`Total Revenue per Customer`) as total_revenue_per_customer,
    MEASURE(`Revenue MoM Difference`) as revenue_mom_difference
  FROM
    dkushari_uc.metric_view_schema.demo_orders_metric_view
  WHERE date_format(`Order Month Date`, 'MMMM') = :month
  GROUP BY order_status;


order_status,order_count,total_revenue,total_revenue_prior_month,total_revenue_per_customer,revenue_mom_difference
Open,293511,43353485331.60,14618401602.72,199902.640400603118,28735083728.88
Processing,65910,12022198561.63,9654834249.67,196136.692415857737,2367364311.96
Fulfilled,295610,43630073904.62,4696391540.49,200382.457055949002,38933682364.13


In [0]:
%sql
SELECT
    `Order Month Date`,
    `Order Status`,
    MEASURE(`Order Count`) as order_count,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Month`) as total_revenue_prior_month,
    MEASURE(`Total Revenue per Customer`) as total_revenue_per_customer,
    MEASURE(`Revenue MoM Difference`) as revenue_mom_difference
  FROM
    dkushari_uc.metric_view_schema.demo_orders_metric_view
  WHERE date_format(`Order Month Date`, 'MMMM') = :month
  GROUP BY ALL
  ORDER BY `Order Month Date` ASC
 

Order Month Date,Order Status,order_count,total_revenue,total_revenue_prior_month,total_revenue_per_customer,revenue_mom_difference
1992-04-01T00:00:00.000Z,Fulfilled,93774,14154839483.68,14471982842.46,167022.696508236183,-317143358.78
1993-04-01T00:00:00.000Z,Fulfilled,93450,14125875685.61,14641206574.63,167243.357276085387,-515330889.02
1994-04-01T00:00:00.000Z,Fulfilled,93739,14191363315.44,14584304371.16,167319.412792869270,-392941055.72
1995-04-01T00:00:00.000Z,Fulfilled,14647,1157995419.89,4696391540.49,80405.181217192057,-3538396120.60
1995-04-01T00:00:00.000Z,Open,12927,963981022.70,223277879.71,75630.081806056802,740703142.99
1995-04-01T00:00:00.000Z,Processing,65910,12022198561.63,9654834249.67,196136.692415857737,2367364311.96
1996-04-01T00:00:00.000Z,Open,93197,14082461114.97,14564998783.77,167075.516265304670,-482537668.80
1997-04-01T00:00:00.000Z,Open,93608,14159948573.95,14597537077.94,167482.182172426845,-437588503.99
1998-04-01T00:00:00.000Z,Open,93779,14147094619.98,14618401602.72,166830.913336006321,-471306982.74


In [0]:
%sql
SELECT
    `Order Status` as order_status, 
    MEASURE(`Order Count`) as order_count,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Year`) as total_revenue_prior_year
  FROM
    dkushari_uc.metric_view_schema.demo_orders_metric_view
  WHERE date_format(`Order Year`, 'yyyy') = :year -- for year = 1995
  GROUP BY ALL
  ORDER BY 1;

order_status,order_count,total_revenue,total_revenue_prior_year
Fulfilled,237791,32694715919.99,172079589599.59
Open,707328,103604354788.78,null
Processing,191189,35333603011.43,null


In [0]:
%sql
SELECT
    `Order Status`, 
    date_format(`Order Year`, 'yyyy') order_year,
    MEASURE(`Order Count`) as order_count,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Year`) as total_revenue_prior_year
  FROM
    dkushari_uc.metric_view_schema.demo_orders_metric_view
  --WHERE date_format(`Order Year`, 'yyyy') = :year
  GROUP BY ALL
  ORDER BY order_year;


Order Status,order_year,order_count,total_revenue,total_revenue_prior_year
Fulfilled,1992,1139873,172177707475.98,null
Fulfilled,1993,1138893,172229906369.71,172177707475.98
Fulfilled,1994,1137944,172079589599.59,172229906369.71
Open,1995,707328,103604354788.78,null
Processing,1995,191189,35333603011.43,null
Fulfilled,1995,237791,32694715919.99,172079589599.59
Open,1996,1141556,172605588830.66,103604354788.78
Open,1997,1137325,171861381581.98,172605588830.66
Open,1998,668101,100852367668.13,171861381581.98


In [0]:
%sql   
SELECT
    date_format(`Order Year`, 'yyyy') order_year,
    MEASURE(`Order Count`) as order_count,
    MEASURE(`Total Revenue`) as total_revenue,
    MEASURE(`Total Revenue Prior Year`) as total_revenue_prior_year
  FROM
    dkushari_uc.metric_view_schema.demo_orders_metric_view
  GROUP BY ALL
  ORDER BY order_year;

order_year,order_count,total_revenue,total_revenue_prior_year
1992,1139873,172177707475.98,null
1993,1138893,172229906369.71,172177707475.98
1994,1137944,172079589599.59,172229906369.71
1995,1136308,171632673720.20,172079589599.59
1996,1141556,172605588830.66,171632673720.20
1997,1137325,171861381581.98,172605588830.66
1998,668101,100852367668.13,171861381581.98


In [0]:
%sql
select date_format(DATE_TRUNC('YEAR', o_orderdate), 'yyyy') order_year, sum(o_totalprice) yearly_revenue
from   dkushari_uc.metric_view_demo.orders
group by order_year
order by 1;

order_year,yearly_revenue
1992,172177707475.98
1993,172229906369.71
1994,172079589599.59
1995,171632673720.20
1996,172605588830.66
1997,171861381581.98
1998,100852367668.13


In [0]:
%sql
SELECT
 `Order Priority`,
 MEASURE(`Order Count`) as order_count,
 MEASURE(`Total Revenue`) as total_revenue,
 MEASURE(`Total Revenue per Customer`) as total_revenue_per_customer
FROM
 demo_orders_metric_view
GROUP BY `Order Priority`
ORDER BY 1 ASC
limit 10;

Order Priority,order_count,total_revenue,total_revenue_per_customer
HIGH,1499192,226785170189.44,491212.972919659334
LOW,1499717,226628507182.19,490647.281065237488
MEDIUM,1498710,226291403383.43,490451.551999874294
NOT SPECIFIED,1501281,226828528845.12,491240.917831708342
URGENT,1501100,226905605646.07,491472.769628794524


In [0]:
%sql
use dkushari_uc.metric_view_demo;
SELECT orders.o_orderkey,
       orders.o_orderdate, 
       sum(l_quantity) as total_quantity, 
       sum(l_extendedprice) price
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
group by ALL
order by orders.o_orderkey
limit 10;

o_orderkey,o_orderdate,total_quantity,price
1,1996-01-02,145.00,211959.15
2,1996-12-01,38.00,71433.16
3,1993-10-14,177.00,243078.62
4,1995-10-11,30.00,33424.50
5,1994-07-30,91.00,155673.40
6,1992-02-21,37.00,43517.18
7,1996-01-10,173.00,259926.11
32,1995-07-16,116.00,153437.80
33,1993-10-27,109.00,177865.35
34,1998-07-21,41.00,55025.04


In [0]:
%sql
USE dkushari_uc.metric_view_schema;
DROP VIEW IF EXISTS demo_lineitem_metric_view;
CREATE VIEW `demo_lineitem_metric_view` 
(
 `order_date` COMMENT "The order date",
 `order_key` COMMENT "Order key", 
 `total_quantity` COMMENT "Total quantity", 
 `total_price` COMMENT "Total price"
 )  
WITH METRICS
LANGUAGE YAML 
AS $$ 
version: 1.1

source: select * from dkushari_uc.metric_view_demo_setup.lineitem

joins:
 - name: orders
   source: select * from dkushari_uc.metric_view_demo_setup.orders
   on: o_orderkey = l_orderkey

dimensions:
 - name: order_date
   expr: orders.o_orderdate 
 - name: order_key
   expr: orders.o_orderkey 

measures:
 - name: total_quantity
   expr: sum(l_quantity)
 - name: total_price
   expr: sum(l_extendedprice)
$$


result


In [0]:
%sql
DESCRIBE TABLE EXTENDED demo_lineitem_metric_view;

col_name,data_type,comment,metadata
order_date,date,The order date,null
order_key,bigint,Order key,null
total_quantity,"decimal(28,2) measure",Total quantity,null
total_price,"decimal(28,2) measure",Total price,null
,,,
# Detailed Table Information,,,
Catalog,dkushari_uc,,
Database,metric_view_schema,,
Table,demo_lineitem_metric_view,,
Owner,dipankar.kushari@databricks.com,,


In [0]:
%sql
DESCRIBE TABLE EXTENDED demo_lineitem_metric_view AS JSON;

json_metadata


In [0]:
%sql
SHOW VIEWS IN dkushari_uc.metric_view_schema;

namespace,viewName,isTemporary,isMaterialized,isMetric
metric_view_schema,demo_lineitem_metric_view,false,false,true
metric_view_schema,demo_mv_refer_mv,false,false,true
metric_view_schema,demo_order_mat_metric_view,false,true,true
metric_view_schema,demo_orders_metric_view,false,false,true
metric_view_schema,order_metric,false,false,true
,_sqldf,true,false,false


In [0]:
%sql
use dkushari_uc.metric_view_schema;
SELECT order_key,
       order_date,
       MEASURE(total_quantity),
       MEASURE(total_price)
FROM demo_lineitem_metric_view
group by ALL
order by order_key
limit 10;

order_key,order_date,MEASURE(total_quantity),MEASURE(total_price)
1,1996-01-02,145.00,211959.15
2,1996-12-01,38.00,71433.16
3,1993-10-14,177.00,243078.62
4,1995-10-11,30.00,33424.50
5,1994-07-30,91.00,155673.40
6,1992-02-21,37.00,43517.18
7,1996-01-10,173.00,259926.11
32,1995-07-16,116.00,153437.80
33,1993-10-27,109.00,177865.35
34,1998-07-21,41.00,55025.04


In [0]:
%sql
use dkushari_uc.metric_view_schema;
ALTER VIEW `demo_lineitem_metric_view` rename to `lineitem_order_metric_view`;

In [0]:
%sql
show views in dkushari_uc.metric_view_schema;

namespace,viewName,isTemporary,isMaterialized,isMetric
metric_view_schema,demo_mv_refer_mv,false,false,true
metric_view_schema,demo_order_mat_metric_view,false,true,true
metric_view_schema,demo_orders_metric_view,false,false,true
metric_view_schema,lineitem_order_metric_view,false,false,true
metric_view_schema,order_metric,false,false,true
,_sqldf,true,false,false


In [0]:
%sql
DESCRIBE TABLE EXTENDED lineitem_order_metric_view;

col_name,data_type,comment,metadata
order_date,date,The order date,null
order_key,bigint,Order key,null
total_quantity,"decimal(28,2) measure",Total quantity,null
total_price,"decimal(28,2) measure",Total price,null
,,,
# Detailed Table Information,,,
Catalog,dkushari_uc,,
Database,metric_view_schema,,
Table,lineitem_order_metric_view,,
Owner,dipankar.kushari@databricks.com,,
